In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("01_Ingest_MovieLens_To_HDFS")
    .master("spark://spark-master:7077")
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020")
    .config("spark.executor.memory", "3g")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "64")
    .getOrCreate()
)

LOCAL_RAW = "/workspace/data/raw/ml-25m"
HDFS_RAW = "hdfs://namenode:8020/netflix-recsys/raw/ml-25m"

In [ ]:
ratings = spark.read.csv(f"{LOCAL_RAW}/ratings.csv", header=True, inferSchema=True)
movies = spark.read.csv(f"{LOCAL_RAW}/movies.csv", header=True, inferSchema=True)
tags = spark.read.csv(f"{LOCAL_RAW}/tags.csv", header=True, inferSchema=True)
links = spark.read.csv(f"{LOCAL_RAW}/links.csv", header=True, inferSchema=True)

In [ ]:
print("ratings:", ratings.count())
print("movies:", movies.count())
print("tags:", tags.count())
print("links:", links.count())

ratings.printSchema()
movies.printSchema()

In [ ]:
ratings.write.mode("overwrite").parquet(f"{HDFS_RAW}/ratings")
movies.write.mode("overwrite").parquet(f"{HDFS_RAW}/movies")
tags.write.mode("overwrite").parquet(f"{HDFS_RAW}/tags")
links.write.mode("overwrite").parquet(f"{HDFS_RAW}/links")

In [ ]:
for path in [
    "hdfs://namenode:8020/netflix-recsys/bronze",
    "hdfs://namenode:8020/netflix-recsys/silver",
    "hdfs://namenode:8020/netflix-recsys/gold",
    "hdfs://namenode:8020/netflix-recsys/models",
    "hdfs://namenode:8020/netflix-recsys/outputs",
]:
    spark._jvm.org.apache.hadoop.fs.FileSystem.get(
        spark._jsc.hadoopConfiguration()
    ).mkdirs(
        spark._jvm.org.apache.hadoop.fs.Path(path)
    )